# 한이음 NLU 공정 비교 실험  
## 독립적인 4개 koELECTRA Classifier

- 입력: `text`
- 출력: `intent`, `menu`, `temperature`, `quantity`
- 구조: 각 task마다 독립적인 `monologg/koelectra-small-v3-discriminator`
- 데이터 재분할·정규화·중복 제거 없음
- best epoch: validation `frame_accuracy` 우선, 동률이면 `intent_macro_f1`
- test는 best model 선택이 끝난 뒤 정확히 한 번 최종 평가

## 1. 호환 패키지 설치

Google Colab T4 런타임 기준 설치 셀입니다. Colab에 함께 설치된 `umap-learn`과 `hdbscan`이 `scikit-learn>=1.6`을 요구하므로 `scikit-learn==1.6.1`을 사용합니다.  
이 셀 실행 전에 해당 패키지들을 이미 import했다면 **런타임 → 세션 다시 시작** 후 이 셀부터 다시 실행하세요. 아직 import하지 않았다면 재시작 없이 계속 진행해도 됩니다.

In [ ]:
%pip -q install -U \
  "transformers==4.46.3" \
  "accelerate==1.1.1" \
  "scikit-learn==1.6.1" \
  "sentencepiece==0.2.0"

## 2. 데이터 업로드

In [ ]:
from google.colab import files

print("train.csv, valid.csv, test.csv 세 파일을 선택하세요.")
uploaded = files.upload()

required_files = {"train.csv", "valid.csv", "test.csv"}
missing_files = required_files - set(uploaded.keys())
if missing_files:
    raise FileNotFoundError(f"업로드되지 않은 파일: {sorted(missing_files)}")
print("업로드 완료:", sorted(required_files))

## 3. CSV 불러오기 — 파일 수정·재분할 없음

In [ ]:
import pandas as pd

# dtype=str로 읽어 quantity의 숫자/문자열 혼합 문제를 방지한다.
# keep_default_na=False로 빈 값이나 NONE 문자열을 임의의 NaN으로 바꾸지 않는다.
train_df = pd.read_csv("train.csv", dtype=str, keep_default_na=False)
valid_df = pd.read_csv("valid.csv", dtype=str, keep_default_na=False)
test_df  = pd.read_csv("test.csv",  dtype=str, keep_default_na=False)

print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test :", test_df.shape)
display(train_df.head())

## 4. 필수 컬럼 존재 여부만 검사

In [ ]:
REQUIRED_COLUMNS = ["text", "label", "menu", "temperature", "quantity"]

for split_name, df in {
    "train": train_df,
    "valid": valid_df,
    "test": test_df,
}.items():
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"{split_name}.csv 필수 컬럼 누락: {missing}")

print("필수 컬럼 검사 통과")

## 5. import, 실험 설정, random seed 고정

In [ ]:
import os
import gc
import json
import math
import random
import shutil
import zipfile
import platform
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import transformers
import accelerate

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)
from tqdm.auto import tqdm

SEED = 42
EXPERIMENT_NAME = "four_independent_koelectra_classifiers"
MODEL_ARCHITECTURE = "4 independent koELECTRA sequence classifiers"
ENCODER_NAME = "monologg/koelectra-small-v3-discriminator"

EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
MAX_LENGTH = 64
WARMUP_RATIO = 0.1
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

RESULT_DIR = Path("experiment_results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("device:", device)
print("GPU:", gpu_name)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

## 6. LabelEncoder 생성 및 미등록 class 사전 검사

In [ ]:
TASK_COLUMNS = {
    "intent": "label",
    "menu": "menu",
    "temperature": "temperature",
    "quantity": "quantity",
}

# intent는 train 전체로 학습한다.
# menu/temperature/quantity는 정답 intent가 ORDER인 train 행만 학습·평가하므로
# slot label space도 ORDER 행만 사용해 정의한다.
train_order_mask = train_df["label"].eq("ORDER")
valid_order_mask = valid_df["label"].eq("ORDER")
test_order_mask = test_df["label"].eq("ORDER")

if train_order_mask.sum() == 0:
    raise ValueError("train.csv에 ORDER 행이 없습니다.")

label_encoders = {}

intent_le = LabelEncoder()
intent_le.fit(train_df["label"].astype(str))
label_encoders["intent"] = intent_le

for task in ["menu", "temperature", "quantity"]:
    col = TASK_COLUMNS[task]
    le = LabelEncoder()
    le.fit(train_df.loc[train_order_mask, col].astype(str))
    label_encoders[task] = le

def assert_no_unseen_classes(split_name: str, df: pd.DataFrame):
    problems = {}

    intent_known = set(label_encoders["intent"].classes_)
    unseen_intent = sorted(set(df["label"].astype(str)) - intent_known)
    if unseen_intent:
        problems["intent"] = unseen_intent

    order_mask = df["label"].eq("ORDER")
    for task in ["menu", "temperature", "quantity"]:
        col = TASK_COLUMNS[task]
        known = set(label_encoders[task].classes_)
        unseen = sorted(set(df.loc[order_mask, col].astype(str)) - known)
        if unseen:
            problems[task] = unseen

    if problems:
        details = "\n".join(f"- {task}: {classes}" for task, classes in problems.items())
        raise ValueError(
            f"{split_name}.csv에 train.csv에 없던 class가 있습니다.\n{details}"
        )

assert_no_unseen_classes("valid", valid_df)
assert_no_unseen_classes("test", test_df)

label_maps = {
    task: {
        "classes": le.classes_.tolist(),
        "class_to_id": {str(c): int(i) for i, c in enumerate(le.classes_)},
        "id_to_class": {str(i): str(c) for i, c in enumerate(le.classes_)},
    }
    for task, le in label_encoders.items()
}

with open(RESULT_DIR / "label_maps.json", "w", encoding="utf-8") as f:
    json.dump(label_maps, f, ensure_ascii=False, indent=2)

for task, le in label_encoders.items():
    print(f"{task}: {len(le.classes_)} classes -> {le.classes_.tolist()}")

## 7. Tokenizer, Dataset, DataLoader 생성

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
tokenizer.save_pretrained(RESULT_DIR / "tokenizer")

class NluDataset(Dataset):
    def __init__(self, df, tokenizer, label_encoders, max_length=64):
        self.df = df.reset_index(drop=True).copy()
        self.tokenizer = tokenizer
        self.label_encoders = label_encoders
        self.max_length = max_length

        self.intent_ids = label_encoders["intent"].transform(
            self.df["label"].astype(str)
        ).astype(np.int64)

        self.is_order = self.df["label"].eq("ORDER").to_numpy(dtype=np.bool_)

        # 비-ORDER 행의 slot label은 loss/평가에서 사용되지 않는다.
        # Dataset tensor shape 유지를 위해 dummy id 0을 넣는다.
        self.slot_ids = {}
        for task in ["menu", "temperature", "quantity"]:
            col = TASK_COLUMNS[task]
            ids = np.zeros(len(self.df), dtype=np.int64)
            if self.is_order.any():
                ids[self.is_order] = label_encoders[task].transform(
                    self.df.loc[self.is_order, col].astype(str)
                )
            self.slot_ids[task] = ids

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.df.at[idx, "text"]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in encoded.items()}
        item.update({
            "intent_label": torch.tensor(self.intent_ids[idx], dtype=torch.long),
            "menu_label": torch.tensor(self.slot_ids["menu"][idx], dtype=torch.long),
            "temperature_label": torch.tensor(self.slot_ids["temperature"][idx], dtype=torch.long),
            "quantity_label": torch.tensor(self.slot_ids["quantity"][idx], dtype=torch.long),
            "is_order": torch.tensor(self.is_order[idx], dtype=torch.bool),
            "row_index": torch.tensor(idx, dtype=torch.long),
        })
        return item

train_dataset = NluDataset(train_df, tokenizer, label_encoders, MAX_LENGTH)
valid_dataset = NluDataset(valid_df, tokenizer, label_encoders, MAX_LENGTH)
test_dataset  = NluDataset(test_df,  tokenizer, label_encoders, MAX_LENGTH)

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("DataLoader 생성 완료")

## 8. 모델 클래스 — task별 독립 koELECTRA 4개

In [ ]:
class FourIndependentKoElectra(nn.Module):
    """
    intent/menu/temperature/quantity 각각에 독립적인 koELECTRA encoder와
    classification head를 사용한다. 네 모델 사이에는 encoder parameter 공유가 없다.
    """
    def __init__(self, encoder_name, num_labels):
        super().__init__()
        self.intent_model = AutoModelForSequenceClassification.from_pretrained(
            encoder_name, num_labels=num_labels["intent"]
        )
        self.menu_model = AutoModelForSequenceClassification.from_pretrained(
            encoder_name, num_labels=num_labels["menu"]
        )
        self.temperature_model = AutoModelForSequenceClassification.from_pretrained(
            encoder_name, num_labels=num_labels["temperature"]
        )
        self.quantity_model = AutoModelForSequenceClassification.from_pretrained(
            encoder_name, num_labels=num_labels["quantity"]
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        common = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }
        if token_type_ids is not None:
            common["token_type_ids"] = token_type_ids

        return {
            "intent": self.intent_model(**common).logits,
            "menu": self.menu_model(**common).logits,
            "temperature": self.temperature_model(**common).logits,
            "quantity": self.quantity_model(**common).logits,
        }

num_labels = {task: len(le.classes_) for task, le in label_encoders.items()}
model = FourIndependentKoElectra(ENCODER_NAME, num_labels).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total parameters: {total_params:,}")
print(f"trainable parameters: {trainable_params:,}")

## 9. 공통 metric 및 평가 함수

In [ ]:
criterion = nn.CrossEntropyLoss()

def safe_macro_f1(y_true, y_pred):
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))

def compute_metrics_from_ids(
    true_intent, pred_intent,
    true_menu, pred_menu,
    true_temperature, pred_temperature,
    true_quantity, pred_quantity,
):
    true_intent = np.asarray(true_intent)
    pred_intent = np.asarray(pred_intent)
    order_id = int(label_encoders["intent"].transform(["ORDER"])[0])
    order_mask = true_intent == order_id

    intent_correct = pred_intent == true_intent

    metrics = {
        "intent_accuracy": float(accuracy_score(true_intent, pred_intent)),
        "intent_macro_f1": safe_macro_f1(true_intent, pred_intent),
    }

    if order_mask.sum() == 0:
        raise ValueError("평가 데이터에 ORDER 행이 없어 slot 지표를 계산할 수 없습니다.")

    slot_values = {}
    for task, true_ids, pred_ids in [
        ("menu", true_menu, pred_menu),
        ("temperature", true_temperature, pred_temperature),
        ("quantity", true_quantity, pred_quantity),
    ]:
        t = np.asarray(true_ids)[order_mask]
        p = np.asarray(pred_ids)[order_mask]
        metrics[f"{task}_accuracy"] = float(accuracy_score(t, p))
        metrics[f"{task}_macro_f1"] = safe_macro_f1(t, p)
        slot_values[task] = (t, p)

    order_exact = (
        (slot_values["menu"][0] == slot_values["menu"][1])
        & (slot_values["temperature"][0] == slot_values["temperature"][1])
        & (slot_values["quantity"][0] == slot_values["quantity"][1])
    )
    metrics["order_exact_match"] = float(order_exact.mean())

    frame_correct = intent_correct.copy()
    frame_correct[order_mask] = (
        intent_correct[order_mask]
        & (np.asarray(true_menu)[order_mask] == np.asarray(pred_menu)[order_mask])
        & (np.asarray(true_temperature)[order_mask] == np.asarray(pred_temperature)[order_mask])
        & (np.asarray(true_quantity)[order_mask] == np.asarray(pred_quantity)[order_mask])
    )
    metrics["frame_accuracy"] = float(frame_correct.mean())
    return metrics, frame_correct, order_mask

@torch.no_grad()
def evaluate(model, data_loader, desc="Evaluation"):
    model.eval()
    store = {
        "row_index": [],
        "true_intent": [], "pred_intent": [],
        "true_menu": [], "pred_menu": [],
        "true_temperature": [], "pred_temperature": [],
        "true_quantity": [], "pred_quantity": [],
    }

    for batch in tqdm(data_loader, desc=desc, leave=False):
        model_inputs = {
            "input_ids": batch["input_ids"].to(device, non_blocking=True),
            "attention_mask": batch["attention_mask"].to(device, non_blocking=True),
        }
        if "token_type_ids" in batch:
            model_inputs["token_type_ids"] = batch["token_type_ids"].to(
                device, non_blocking=True
            )

        logits = model(**model_inputs)

        store["row_index"].extend(batch["row_index"].cpu().numpy().tolist())
        for task in ["intent", "menu", "temperature", "quantity"]:
            store[f"true_{task}"].extend(batch[f"{task}_label"].cpu().numpy().tolist())
            store[f"pred_{task}"].extend(
                logits[task].argmax(dim=-1).cpu().numpy().tolist()
            )

    # DataLoader shuffle=False지만 row_index 기준으로 한 번 더 정렬해 CSV 일관성을 보장한다.
    order = np.argsort(np.asarray(store["row_index"]))
    for key in store:
        store[key] = np.asarray(store[key])[order]

    metrics, frame_correct, order_mask = compute_metrics_from_ids(
        store["true_intent"], store["pred_intent"],
        store["true_menu"], store["pred_menu"],
        store["true_temperature"], store["pred_temperature"],
        store["true_quantity"], store["pred_quantity"],
    )
    store["frame_correct"] = frame_correct
    store["order_mask"] = order_mask
    return metrics, store

## 10. 학습 함수 및 best validation model 저장

In [ ]:
def build_optimizer(model):
    no_decay = ["bias", "LayerNorm.weight"]
    grouped = [
        {
            "params": [
                p for n, p in model.named_parameters()
                if p.requires_grad and not any(nd in n for nd in no_decay)
            ],
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": [
                p for n, p in model.named_parameters()
                if p.requires_grad and any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.0,
        },
    ]
    return torch.optim.AdamW(grouped, lr=LEARNING_RATE)

optimizer = build_optimizer(model)
total_steps = EPOCHS * len(train_loader)
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

def train_one_epoch(model, data_loader, optimizer, scheduler, epoch):
    model.train()
    running_loss = 0.0
    steps = 0

    progress = tqdm(data_loader, desc=f"Train epoch {epoch}", leave=False)
    for batch in progress:
        optimizer.zero_grad(set_to_none=True)

        model_inputs = {
            "input_ids": batch["input_ids"].to(device, non_blocking=True),
            "attention_mask": batch["attention_mask"].to(device, non_blocking=True),
        }
        if "token_type_ids" in batch:
            model_inputs["token_type_ids"] = batch["token_type_ids"].to(
                device, non_blocking=True
            )

        logits = model(**model_inputs)
        intent_labels = batch["intent_label"].to(device, non_blocking=True)
        is_order = batch["is_order"].to(device, non_blocking=True)

        # Loss 정의:
        # 1) intent CE는 모든 train 행에 적용
        # 2) menu/temperature/quantity CE는 정답 intent가 ORDER인 행에만 적용
        # 3) 네 task loss는 동일 가중치로 합산
        intent_loss = criterion(logits["intent"], intent_labels)
        total_loss = intent_loss

        if is_order.any():
            menu_labels = batch["menu_label"].to(device, non_blocking=True)[is_order]
            temp_labels = batch["temperature_label"].to(device, non_blocking=True)[is_order]
            qty_labels = batch["quantity_label"].to(device, non_blocking=True)[is_order]

            menu_loss = criterion(logits["menu"][is_order], menu_labels)
            temperature_loss = criterion(logits["temperature"][is_order], temp_labels)
            quantity_loss = criterion(logits["quantity"][is_order], qty_labels)
            total_loss = (
                intent_loss + menu_loss + temperature_loss + quantity_loss
            )

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        scheduler.step()

        running_loss += total_loss.item()
        steps += 1
        progress.set_postfix(loss=f"{running_loss / steps:.4f}")

    return running_loss / max(steps, 1)

best_frame = -1.0
best_intent_f1 = -1.0
best_epoch = 0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, epoch)
    valid_metrics, _ = evaluate(model, valid_loader, desc=f"Valid epoch {epoch}")

    row = {"epoch": epoch, "train_loss": train_loss, **valid_metrics}
    history.append(row)
    pd.DataFrame(history).to_csv(
        RESULT_DIR / "validation_history.csv", index=False, encoding="utf-8-sig"
    )

    current_frame = valid_metrics["frame_accuracy"]
    current_intent_f1 = valid_metrics["intent_macro_f1"]
    is_better = (
        current_frame > best_frame
        or (
            math.isclose(current_frame, best_frame, rel_tol=0.0, abs_tol=1e-12)
            and current_intent_f1 > best_intent_f1
        )
    )

    if is_better:
        best_frame = current_frame
        best_intent_f1 = current_intent_f1
        best_epoch = epoch
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "best_validation_frame_accuracy": best_frame,
                "best_validation_intent_macro_f1": best_intent_f1,
                "encoder_name": ENCODER_NAME,
                "num_labels": num_labels,
                "seed": SEED,
            },
            RESULT_DIR / "best_model.pt",
        )

    print(
        f"[Epoch {epoch:02d}] loss={train_loss:.4f} | "
        f"valid frame={current_frame:.4f} | "
        f"valid intent_macro_f1={current_intent_f1:.4f} | "
        f"best_epoch={best_epoch}"
    )

print("학습 완료")
print("best_epoch:", best_epoch)
print("best validation frame_accuracy:", best_frame)

## 11. best model 재로딩 후 test 최종 평가 — 이 셀에서 test를 한 번만 평가

In [ ]:
# best epoch 선택은 이미 valid 결과로 완료되었다.
# 아래 evaluate(test_loader)는 최종 test 평가 1회이다.
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = FourIndependentKoElectra(ENCODER_NAME, num_labels).to(device)
checkpoint = torch.load(
    RESULT_DIR / "best_model.pt",
    map_location=device,
    weights_only=False,
)
model.load_state_dict(checkpoint["model_state_dict"])

test_metrics_core, test_store = evaluate(
    model, test_loader, desc="FINAL TEST (run once)"
)

print(json.dumps(test_metrics_core, ensure_ascii=False, indent=2))

## 12. test 행별 예측 결과 및 오답 분석 CSV 생성

In [ ]:
def inverse(task, ids):
    return label_encoders[task].inverse_transform(np.asarray(ids, dtype=int))

true_intent_text = inverse("intent", test_store["true_intent"])
pred_intent_text = inverse("intent", test_store["pred_intent"])

pred_menu_text = inverse("menu", test_store["pred_menu"])
pred_temperature_text = inverse("temperature", test_store["pred_temperature"])
pred_quantity_text = inverse("quantity", test_store["pred_quantity"])

predictions = pd.DataFrame({
    "text": test_df["text"].astype(str),
    "true_intent": test_df["label"].astype(str),
    "pred_intent": pred_intent_text,
    "true_menu": test_df["menu"].astype(str),
    "pred_menu": pred_menu_text,
    "true_temperature": test_df["temperature"].astype(str),
    "pred_temperature": pred_temperature_text,
    "true_quantity": test_df["quantity"].astype(str),
    "pred_quantity": pred_quantity_text,
})

order_mask = predictions["true_intent"].eq("ORDER")

# 비-ORDER 행에서는 slot 예측 자체가 평가 대상이 아니므로 출력상 NONE으로 고정하고
# correctness/order_exact_match는 공란(pd.NA)으로 둔다.
for col in ["pred_menu", "pred_temperature", "pred_quantity"]:
    predictions.loc[~order_mask, col] = "NONE"

predictions["intent_correct"] = (
    predictions["true_intent"] == predictions["pred_intent"]
)

for task in ["menu", "temperature", "quantity"]:
    correct_col = f"{task}_correct"
    predictions[correct_col] = pd.Series(pd.NA, index=predictions.index, dtype="boolean")
    predictions.loc[order_mask, correct_col] = (
        predictions.loc[order_mask, f"true_{task}"].astype(str).to_numpy()
        == predictions.loc[order_mask, f"pred_{task}"].astype(str).to_numpy()
    )

predictions["order_exact_match"] = pd.Series(
    pd.NA, index=predictions.index, dtype="boolean"
)
predictions.loc[order_mask, "order_exact_match"] = (
    predictions.loc[order_mask, "menu_correct"].fillna(False).astype(bool)
    & predictions.loc[order_mask, "temperature_correct"].fillna(False).astype(bool)
    & predictions.loc[order_mask, "quantity_correct"].fillna(False).astype(bool)
)

predictions["frame_correct"] = predictions["intent_correct"].astype(bool)
predictions.loc[order_mask, "frame_correct"] = (
    predictions.loc[order_mask, "intent_correct"].astype(bool)
    & predictions.loc[order_mask, "order_exact_match"].fillna(False).astype(bool)
)

# JSON metrics와 CSV 집계값의 일치 여부를 강제 검사한다.
csv_metric_checks = {
    "intent_accuracy": float(predictions["intent_correct"].mean()),
    "menu_accuracy": float(predictions.loc[order_mask, "menu_correct"].astype(bool).mean()),
    "temperature_accuracy": float(predictions.loc[order_mask, "temperature_correct"].astype(bool).mean()),
    "quantity_accuracy": float(predictions.loc[order_mask, "quantity_correct"].astype(bool).mean()),
    "order_exact_match": float(predictions.loc[order_mask, "order_exact_match"].astype(bool).mean()),
    "frame_accuracy": float(predictions["frame_correct"].mean()),
}
for key, csv_value in csv_metric_checks.items():
    if not math.isclose(
        csv_value, test_metrics_core[key], rel_tol=0.0, abs_tol=1e-12
    ):
        raise AssertionError(
            f"metric 불일치: {key}, JSON={test_metrics_core[key]}, CSV={csv_value}"
        )

predictions.to_csv(
    RESULT_DIR / "test_predictions.csv", index=False, encoding="utf-8-sig"
)
errors = predictions.loc[~predictions["frame_correct"]].copy()
errors.to_csv(
    RESULT_DIR / "test_errors.csv", index=False, encoding="utf-8-sig"
)

print("test_predictions rows:", len(predictions))
print("test_errors rows:", len(errors))
display(predictions.head())

## 13. task/class별 precision, recall, f1 저장

In [ ]:
class_metric_rows = []

def append_class_metrics(task, true_ids, pred_ids, classes):
    labels = np.arange(len(classes))
    precision, recall, f1, support = precision_recall_fscore_support(
        true_ids,
        pred_ids,
        labels=labels,
        zero_division=0,
    )
    for idx, class_name in enumerate(classes):
        class_metric_rows.append({
            "task": task,
            "class": str(class_name),
            "support": int(support[idx]),
            "precision": float(precision[idx]),
            "recall": float(recall[idx]),
            "f1": float(f1[idx]),
        })

append_class_metrics(
    "intent",
    test_store["true_intent"],
    test_store["pred_intent"],
    label_encoders["intent"].classes_,
)

order_id = int(label_encoders["intent"].transform(["ORDER"])[0])
order_id_mask = test_store["true_intent"] == order_id

for task in ["menu", "temperature", "quantity"]:
    append_class_metrics(
        task,
        test_store[f"true_{task}"][order_id_mask],
        test_store[f"pred_{task}"][order_id_mask],
        label_encoders[task].classes_,
    )

class_metrics_df = pd.DataFrame(class_metric_rows)
class_metrics_df.to_csv(
    RESULT_DIR / "class_metrics.csv", index=False, encoding="utf-8-sig"
)
display(class_metrics_df)

## 14. 실험 설정과 최종 결과 JSON 저장

In [ ]:
experiment_config = {
    "experiment_name": EXPERIMENT_NAME,
    "model_architecture": MODEL_ARCHITECTURE,
    "encoder_name": ENCODER_NAME,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "warmup_ratio": WARMUP_RATIO,
    "gradient_clipping": GRAD_CLIP_NORM,
    "train_rows": int(len(train_df)),
    "valid_rows": int(len(valid_df)),
    "test_rows": int(len(test_df)),
    "intent_classes": label_encoders["intent"].classes_.tolist(),
    "menu_classes": label_encoders["menu"].classes_.tolist(),
    "temperature_classes": label_encoders["temperature"].classes_.tolist(),
    "quantity_classes": label_encoders["quantity"].classes_.tolist(),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "accelerate_version": accelerate.__version__,
    "gpu_name": gpu_name,
}

with open(RESULT_DIR / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(experiment_config, f, ensure_ascii=False, indent=2)

model_size_mb = (RESULT_DIR / "best_model.pt").stat().st_size / (1024 ** 2)

test_metrics = {
    "experiment_name": EXPERIMENT_NAME,
    "model_architecture": MODEL_ARCHITECTURE,
    "encoder_name": ENCODER_NAME,
    "seed": SEED,
    "best_epoch": int(checkpoint["epoch"]),
    "best_validation_frame_accuracy": float(
        checkpoint["best_validation_frame_accuracy"]
    ),
    "intent_accuracy": test_metrics_core["intent_accuracy"],
    "intent_macro_f1": test_metrics_core["intent_macro_f1"],
    "menu_accuracy": test_metrics_core["menu_accuracy"],
    "menu_macro_f1": test_metrics_core["menu_macro_f1"],
    "temperature_accuracy": test_metrics_core["temperature_accuracy"],
    "temperature_macro_f1": test_metrics_core["temperature_macro_f1"],
    "quantity_accuracy": test_metrics_core["quantity_accuracy"],
    "quantity_macro_f1": test_metrics_core["quantity_macro_f1"],
    "order_exact_match": test_metrics_core["order_exact_match"],
    "frame_accuracy": test_metrics_core["frame_accuracy"],
    "model_size_mb": float(model_size_mb),
}

with open(RESULT_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

print(json.dumps(test_metrics, ensure_ascii=False, indent=2))

## 15. 결과 폴더 검증 및 ZIP 압축

In [ ]:
expected_paths = [
    RESULT_DIR / "best_model.pt",
    RESULT_DIR / "tokenizer",
    RESULT_DIR / "experiment_config.json",
    RESULT_DIR / "label_maps.json",
    RESULT_DIR / "validation_history.csv",
    RESULT_DIR / "test_metrics.json",
    RESULT_DIR / "test_predictions.csv",
    RESULT_DIR / "test_errors.csv",
    RESULT_DIR / "class_metrics.csv",
]

missing_outputs = [str(p) for p in expected_paths if not p.exists()]
if missing_outputs:
    raise FileNotFoundError(f"결과 파일 누락: {missing_outputs}")

zip_path = Path("experiment_results.zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in RESULT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=path.as_posix())

print("ZIP 생성 완료:", zip_path)
print("크기(MB):", round(zip_path.stat().st_size / (1024 ** 2), 2))
print("\n결과 파일:")
for path in sorted(RESULT_DIR.rglob("*")):
    if path.is_file():
        print("-", path)

## 16. Colab에서 컴퓨터로 ZIP 다운로드

In [ ]:
from google.colab import files
files.download("experiment_results.zip")